# Wikidata Ogham Sites — Interactive Map

## About this notebook

This notebook fetches Ogham-stone records from
[Wikidata](https://www.wikidata.org) and displays their find-spots on
an interactive [Leaflet](https://leafletjs.com/) map (via
[`folium`](https://python-visualization.github.io/folium/)) with an
OpenStreetMap basemap. It is a companion to
`wikidata-ogham-sites.ipynb` (bar-chart overview of the same dataset)
and part of an Open Educational Resource series on knowledge graphs
and linked open data. A browser-executable variant of this notebook
is available as `wikidata-ogham-sites-map-live.qmd`.

### What you'll learn

- How to extract **geographic coordinates** from a Wikidata SPARQL
  response. Coordinates in Wikidata are stored as
  [WKT](https://en.wikipedia.org/wiki/Well-known_text_representation_of_geometry)
  `Point(lon lat)` literals — a string format that needs a small
  parser before the values can be plotted.
- How to build an **interactive web map** from Python using
  `folium`, which wraps Leaflet in a Python-friendly API and renders
  directly inside Jupyter.

### Data-context notes

The query in this notebook only returns records that *have*
coordinates (`?item wdt:P625 ?geo`, not `OPTIONAL`). This is the
right call for a map — records without coordinates cannot be plotted
— but it also filters out a non-trivial share of stones, depending on
how well the data is curated at the time you run it. The overview
notebook makes the opposite choice, making coordinates optional, to
give a complete picture of counts per county. The difference between
those two result sets is itself an instructive thing to look at.

A second subtlety: Wikidata stores coordinates in WGS 84
(`EPSG:4326`), the same coordinate reference system Leaflet and
OpenStreetMap expect. No reprojection is needed. If you later query
an endpoint that uses a different projection (e.g. some national
cultural-heritage registries), you will need to reproject
(`gdf.to_crs(...)` in `geopandas`).

### Tooling notes

- **`SPARQLWrapper`** to query Wikidata. The browser-executable
  variant uses `pyodide.http.pyfetch` instead.
- **`folium`** to build the interactive map. `folium` is a Python
  wrapper around Leaflet: you describe the map in Python, and the
  library emits the HTML + JavaScript Jupyter needs to render it
  inline. The browser variant of this notebook cannot use `folium`
  (it expects to write HTML files locally) and instead builds the
  Leaflet map directly via a small `js.eval()` bridge.
- We deliberately avoid `geopandas` + `contextily` for this example.
  They are excellent tools for publication-quality static maps, but
  heavier to install (GDAL, GEOS, PROJ) and not needed for this
  pattern. If you want static maps with basemaps, they are the
  right choice.

### Requirements

```bash
pip install SPARQLWrapper pandas folium
```

## Step 1 — Defining the SPARQL query

We ask Wikidata for every Ogham stone, its find-spot, its county, and
its coordinate location. The structure is almost identical to the
overview notebook, with the one difference noted above: coordinates
are required.

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import folium
from folium.plugins import HeatMap

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
USER_AGENT = "OER-Notebook/1.0 (NFDI4Objects)"

SPARQL_QUERY = """
SELECT ?item ?itemLabel ?geo ?site ?siteLabel ?county ?countyLabel WHERE {
  ?item wdt:P31 wd:Q2016147.
  ?item wdt:P189 ?site.
  ?site wdt:P31 wd:Q72617071.
  ?item wdt:P189 ?county.
  ?county wdt:P31 wd:Q179872.
  ?item wdt:P625 ?geo.
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
"""

def fetch_wikidata(query, endpoint=SPARQL_ENDPOINT):
    """Send a SPARQL query to the given endpoint and return the raw bindings."""
    sparql = SPARQLWrapper(endpoint, agent=USER_AGENT)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    return sparql.queryAndConvert()["results"]["bindings"]

def parse_wkt_point(wkt):
    """Parse a WKT 'Point(lon lat)' literal into a (lat, lon) tuple."""
    if not wkt:
        return (None, None)
    try:
        coords = wkt.replace("Point(", "").replace(")", "").split()
        lon, lat = float(coords[0]), float(coords[1])
        return (lat, lon)
    except (ValueError, IndexError):
        return (None, None)

print("✓ Helpers defined.")

## Step 2 — Loading the data

The `parse_wkt_point` helper defined above converts each `Point(lon lat)`
literal into a `(lat, lon)` tuple. Note the ordering: WKT is
`lon lat`, but Leaflet (and most mapping libraries) expect
`lat, lon`. Getting this the wrong way round is one of the most
common mistakes when plotting spatial data — the kind of bug where
everything "works" but all your points end up in the wrong
hemisphere.

In [ ]:
results = fetch_wikidata(SPARQL_QUERY)

rows = []
for r in results:
    lat, lon = parse_wkt_point(r.get("geo", {}).get("value"))
    rows.append({
        "item":        r["item"]["value"],
        "itemLabel":   r["itemLabel"]["value"],
        "site":        r["site"]["value"],
        "siteLabel":   r["siteLabel"]["value"],
        "county":      r["county"]["value"],
        "countyLabel": r["countyLabel"]["value"],
        "latitude":    lat,
        "longitude":   lon,
    })

df = pd.DataFrame(rows).dropna(subset=["latitude", "longitude"])
print(
    f"✓ {len(df)} records with coordinates loaded "
    f"({df['itemLabel'].nunique()} unique Ogham stones "
    f"across {df['countyLabel'].nunique()} counties)."
)
df.head()

## Step 3 — Building an interactive folium map

`folium` lets us describe the map in Python. We build two layers —
county-coloured markers with pop-ups and a density heatmap — and add
a layer-control so readers can toggle between them.

In [ ]:
assert not df.empty, "⚠ DataFrame is empty — please run the loading cell first."

# One colour per county, reused consistently across visualisations.
palette = [
    "#e6194B", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#469990", "#9A6324", "#800000",
    "#808000", "#000075", "#e6beff", "#fabed4", "#aaffc3",
    "#fffac8", "#dcbeff", "#ffd8b1", "#a9a9a9", "#ffe119",
]
counties = sorted(df["countyLabel"].unique())
county_colors = {c: palette[i % len(palette)] for i, c in enumerate(counties)}

# Initialise the map, centred roughly on Ireland.
centre = [df["latitude"].mean(), df["longitude"].mean()]
m = folium.Map(location=centre, zoom_start=7, tiles="OpenStreetMap")

# --- Layer 1: county-coloured markers with pop-ups -----------------------
marker_layer = folium.FeatureGroup(name="Markers by county", show=True)
for _, row in df.iterrows():
    popup_html = (
        f"<b>{row['itemLabel']}</b><br>"
        f"Site: {row['siteLabel']}<br>"
        f"County: {row['countyLabel']}<br>"
        f'<a href="{row["item"]}" target="_blank" rel="noopener">Wikidata entry ↗</a>'
    )
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        color=county_colors[row["countyLabel"]],
        weight=1.5,
        fill=True,
        fill_color=county_colors[row["countyLabel"]],
        fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=300),
    ).add_to(marker_layer)
marker_layer.add_to(m)

# --- Layer 2: heatmap ----------------------------------------------------
heat_layer = folium.FeatureGroup(name="Heatmap", show=False)
HeatMap(
    data=df[["latitude", "longitude"]].values.tolist(),
    radius=22, blur=18, max_zoom=11,
).add_to(heat_layer)
heat_layer.add_to(m)

# --- Layer control + fit bounds ------------------------------------------
folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[df["latitude"].min(), df["longitude"].min()],
              [df["latitude"].max(), df["longitude"].max()]])

# --- Legend (injected as raw HTML) ---------------------------------------
legend_rows = "<br>".join(
    f'<span style="display:inline-block;width:10px;height:10px;'
    f'background:{county_colors[c]};margin-right:6px;border-radius:50%"></span>{c}'
    for c in counties
)
legend_html = f"""
<div style="position: fixed; bottom: 30px; right: 30px; z-index: 9999;
            background: rgba(255,255,255,0.92); padding: 8px 10px;
            border-radius: 4px; font: 12px/1.4 system-ui, sans-serif;
            max-height: 220px; overflow-y: auto;
            box-shadow: 0 1px 4px rgba(0,0,0,0.2);">
  <b>County</b><br>{legend_rows}
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m

### Exporting the map as a standalone HTML file

`folium` maps can be saved as standalone HTML files. This is handy
for embedding in static websites, sharing with colleagues who do not
run Python, or preserving a snapshot of the map at a given point in
time (the underlying Wikidata can change from day to day).

In [ ]:
m.save("ogham_sites_map.html")
print("✓ Saved map to ogham_sites_map.html")

## Step 4 — Exploring the data

The cells below are a free playground — filter the DataFrame by
county, compute per-county aggregates, or extend the query yourself.

In [ ]:
# Example: list every Ogham stone from a given county, with coordinates.
# Change the filter below to explore other counties.
county_filter = "County Kerry"

subset = (
    df[df["countyLabel"] == county_filter]
      [["itemLabel", "siteLabel", "latitude", "longitude"]]
      .drop_duplicates()
      .reset_index(drop=True)
)
print(f"Ogham stones in {county_filter}: {len(subset)}")
subset

In [ ]:
# Per-county summary: number of stones and the geographic centroid of
# their find-spots. The centroid is a quick sanity check — it should
# fall inside the county on the map.
summary = (
    df.groupby("countyLabel")
      .agg(
          n_stones=("itemLabel", "nunique"),
          n_sites=("siteLabel", "nunique"),
          mean_lat=("latitude", "mean"),
          mean_lon=("longitude", "mean"),
      )
      .sort_values("n_stones", ascending=False)
)
summary

---

*Part of an Open Educational Resource series on knowledge graphs and
linked open data, produced in the context of
[NFDI4Objects](https://www.nfdi4objects.net/).*